# Banking Intent — Local Training & Evaluation

This notebook is configured to run on your local machine using a `.env` file for credentials.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from the .env file located one level up
load_dotenv(".env")

# Set variables for Hugging Face and LangSmith
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("YOUR_LANGSMITH_API_KEY", "")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "banking-intent-unsloth"
print("Environment variables loaded from .env")

Environment variables loaded from .env


In [2]:
# === CONFIG ===
HF_REPO_ID = "TQZinh/banking-intent-unsloth"
WORKING_DIR = os.getcwd()
CHECKPOINT_DIR = os.path.join(WORKING_DIR, "outputs", "checkpoint")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Working Directory: {WORKING_DIR}")
print(f"Checkpoint Directory: {CHECKPOINT_DIR}")

Working Directory: c:\Users\VINH\OneDrive - VNU-HCMUS\Attachments\Desktop\YEAR 3\ƯDNLP\lab_2\banking-intent-unsloth
Checkpoint Directory: c:\Users\VINH\OneDrive - VNU-HCMUS\Attachments\Desktop\YEAR 3\ƯDNLP\lab_2\banking-intent-unsloth\outputs\checkpoint


## 1. Prepare data

In [3]:
os.chdir(os.path.join(WORKING_DIR, "scripts"))
!python preprocess_data.py
print("Data preprocessing complete.")

Loading BANKING77 dataset from Hugging Face (parquet)...
Creating representative training subset (5,000 samples)...
Final Train size: 5000
Final Test size:  3080
Saved train.csv and test.csv to the sample_data directory.
Data preprocessing complete.


## 2. Configure HF repo for checkpoint push

In [4]:
import yaml

config_path = f"{WORKING_DIR}/configs/train.yaml"
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

config['hub_model_id'] = HF_REPO_ID
config['output_dir'] = CHECKPOINT_DIR

with open(config_path, 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print(f"hub_model_id  : {HF_REPO_ID}")
print(f"output_dir    : {CHECKPOINT_DIR}")


hub_model_id  : TQZinh/banking-intent-unsloth
output_dir    : c:\Users\VINH\OneDrive - VNU-HCMUS\Attachments\Desktop\YEAR 3\ƯDNLP\lab_2\banking-intent-unsloth\outputs\checkpoint


## 3. Train

If this cell is re-run after a session restart, it will **automatically resume** from the latest local checkpoint (if any), or pull from Hub if disk was wiped.

In [5]:
import glob
from huggingface_hub import snapshot_download

output_dir = f"{WORKING_DIR}/outputs/checkpoint"
os.makedirs(output_dir, exist_ok=True)

has_local_checkpoint = bool(glob.glob(os.path.join(output_dir, "checkpoint-*")))

if not has_local_checkpoint:
    try:
        print(f"No local checkpoint. Trying to restore from Hub: {HF_REPO_ID}")
        snapshot_download(
            repo_id=HF_REPO_ID,
            local_dir=output_dir,
            token=os.environ["HF_TOKEN"],
        )
        print("Restored from Hub.")
    except Exception as e:
        print(f"Hub restore skipped ({e}) — starting fresh.")
else:
    latest = sorted(glob.glob(os.path.join(output_dir, "checkpoint-*")))[-1]
    print(f"Local checkpoint found: {latest}")

No local checkpoint. Trying to restore from Hub: TQZinh/banking-intent-unsloth


c:\Users\VINH\miniconda3\envs\manga_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 95.75it/s]

Restored from Hub.


In [6]:
os.chdir(os.path.join(WORKING_DIR, "scripts"))
%run train.py
print("Training process finished.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0427 11:08:41.589000 22300 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
Logged in to HuggingFace Hub.
No checkpoint found — starting fresh.
Loading model: unsloth/Qwen3-4B-unsloth-bnb-4bit
==((====))==  Unsloth 2026.4.6: Fast Qwen3 patching. Transformers: 4.57.0.
   \\   /|    NVIDIA GeForce RTX 3070 Laptop GPU. Num GPUs = 1. Max memory: 8.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.1+cu118. CUDA: 8.6. CUDA Toolkit: 11.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/Qwen3-4B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.6 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Loading data from ../sample_data/train.csv...


...

Saving final adapter to c:\Users\VINH\OneDrive - VNU-HCMUS\Attachments\Desktop\YEAR 3\ƯDNLP\lab_2\banking-intent-unsloth\outputs\checkpoint...
Pushing final adapter to Hub: TQZinh/banking-intent-unsloth


Processing Files (1 / 1): 100%|██████████|  132MB /  132MB, 6.37MB/s  
New Data Upload: 100%|██████████|  132MB /  132MB, 6.37MB/s  


Saved model to https://huggingface.co/TQZinh/banking-intent-unsloth


Processing Files (1 / 1): 100%|██████████| 11.4MB / 11.4MB,  0.00B/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  


Done.
Training process finished.


## 4. Quick sanity check — predict one sample

In [3]:
import sys
import yaml
import os

sys.path.insert(0, f"{WORKING_DIR}/scripts")

# Tương tác với inference.yaml
infer_config_path = f"{WORKING_DIR}/configs/inference.yaml"
with open(infer_config_path, 'r', encoding='utf-8') as f:
    infer_config = yaml.safe_load(f)

infer_config['langsmith_api_key'] = os.environ.get("LANGCHAIN_API_KEY", "")
infer_config['langsmith_project'] = os.environ.get("LANGCHAIN_PROJECT", "banking-intent-unsloth")
infer_config['model_path'] = CHECKPOINT_DIR

with open(infer_config_path, 'w', encoding='utf-8') as f:
    yaml.dump(infer_config, f, default_flow_style=False, allow_unicode=True)

print("Updated inference.yaml with LangSmith credentials and model path.\n")

from inference import IntentClassification

clf = IntentClassification(infer_config_path, mode="finetuned")
test_input = "I am still waiting on my card?"
result = clf.predict(test_input)
print(result)
print("The right one: card_arrival")


Updated inference.yaml with LangSmith credentials and model path.

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\VINH\miniconda3\envs\manga_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0427 21:43:38.691000 25364 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
[FINETUNED] Loading model: C:\Users\VINH\OneDrive - VNU-HCMUS\Attachments\Desktop\YEAR 3\ƯDNLP\lab_2\banking-intent-unsloth\outputs\checkpoint
==((====))==  Unsloth 2026.4.6: Fast Qwen3 patching. Transformers: 4.57.0.
   \\   /|    NVIDIA GeForce RTX 3070 Laptop GPU. Num GPUs = 1. Max memory: 8.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.1+cu118. CUDA: 8.6. CUDA Toolkit: 11.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.4.6 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


LangSmith tracing enabled — project: banking-intent-unsloth
{'input': 'I am still waiting on my card?', 'raw_output': 'Card arrival', 'label': 'card_arrival'}
The right one: card_arrival


In [5]:
import gc
import torch

# Xoá model sanity check cũ nếu đang tồn tại
if 'clf' in globals():
    del clf
    
gc.collect()
torch.cuda.empty_cache()
import sys, gc, yaml
import torch
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

sys.path.insert(0, f"{WORKING_DIR}/scripts")
from inference import IntentClassification

with open(f"{WORKING_DIR}/configs/inference.yaml") as f:
    config = yaml.safe_load(f)
batch_size = config.get("batch_size", 8)

test_df = pd.read_csv(f"{WORKING_DIR}/sample_data/test.csv")

# Stratified 200 samples: 3 per intent (77*3=231) then subsample to 200
sample_df = (
    test_df.groupby("intent_name", group_keys=False)
    .sample(n=3, random_state=42)
    .sample(n=200, random_state=42)
    .reset_index(drop=True)
)
print(f"Sampled {len(sample_df)} rows across {sample_df['intent_name'].nunique()} intents")


def run_eval(clf, df, label, print_per_sample=True):
    texts = df["text"].tolist()
    y_true = df["intent_name"].tolist()
    y_pred = []
    for start in tqdm(range(0, len(texts), batch_size), desc=label):
        results = clf.predict_batch(texts[start:start + batch_size])
        for result, true_label in zip(results, y_true[start:start + batch_size]):
            y_pred.append(result["label"])
            if print_per_sample:
                status = "OK" if result["label"] == true_label else "MISS"
                print(f"  [{status}] true={true_label}  pred={result['label']}", flush=True)
    acc = accuracy_score(y_true, y_pred)
    print(f"\n>>> {label} accuracy: {acc:.4f} ({int(acc*len(df))}/{len(df)})\n", flush=True)
    print(classification_report(y_true, y_pred, digits=4), flush=True)
    return acc


def free_model(clf):
    del clf.model
    del clf.tokenizer
    del clf
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM freed: {torch.cuda.memory_allocated()/1e9:.2f}GB used", flush=True)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\VINH\miniconda3\envs\manga_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0427 16:44:52.925000 27036 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
Sampled 200 rows across 77 intents


## 5. Evaluate — quick test (200 samples)

Stratified sample across intents to verify the pipeline before running the full evaluation.

In [5]:
print("=" * 60)
print("EVALUATE — 200 samples")
print("=" * 60)

results = {}

clf = IntentClassification(f"{WORKING_DIR}/configs/inference.yaml", mode="zero_shot")
results["zero_shot"] = run_eval(clf, sample_df, "zero_shot")
free_model(clf)

clf = IntentClassification(f"{WORKING_DIR}/configs/inference.yaml", mode="finetuned")
results["finetuned"] = run_eval(clf, sample_df, "finetuned")
free_model(clf)

print("=== SUMMARY ===")
for mode, acc in results.items():
    print(f"  {mode:<12} {acc:.4f}  ({acc*100:.2f}%)")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\VINH\miniconda3\envs\manga_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0427 14:22:40.684000 2480 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
Sampled 200 rows across 77 intents
EVALUATE — 200 samples
[ZERO_SHOT] Loading model: unsloth/Qwen3-4B-unsloth-bnb-4bit
==((====))==  Unsloth 2026.4.6: Fast Qwen3 patching. Transformers: 4.57.0.
   \\   /|    NVIDIA GeForce RTX 3070 Laptop GPU. Num GPUs = 1. Max memory: 8.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.1+cu118. CUDA: 8.6. CUDA Toolkit: 11.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/Qwen3-4B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
LangSmith tracing enabled — project: banking-intent-unsloth


zero_shot:   0%|          | 0/25 [00:00<?, ?it/s]

  [OK] true=virtual_card_not_working  pred=virtual_card_not_working
  [OK] true=change_pin  pred=change_pin
  [OK] true=apple_pay_or_google_pay  pred=apple_pay_or_google_pay
  [MISS] true=top_up_by_bank_transfer_charge  pred=transfer_fee_charged
  [OK] true=automatic_top_up  pred=automatic_top_up
  [OK] true=transfer_not_received_by_recipient  pred=transfer_not_received_by_recipient
  [MISS] true=beneficiary_not_allowed  pred=transfer_not_received_by_recipient
  [OK] true=transfer_into_account  pred=transfer_into_account


...

  [MISS] true=wrong_amount_of_cash_received  pred=declined_cash_withdrawal
  [OK] true=virtual_card_not_working  pred=virtual_card_not_working
  [MISS] true=top_up_by_bank_transfer_charge  pred=receiving_money
  [OK] true=card_swallowed  pred=card_swallowed
  [OK] true=card_payment_fee_charged  pred=card_payment_fee_charged
  [OK] true=direct_debit_payment_not_recognised  pred=direct_debit_payment_not_recognised
  [OK] true=balance_not_updated_after_cheque_or_cash_deposit  pred=balance_not_updated_after_cheque_or_cash_deposit
  [OK] true=card_swallowed  pred=card_swallowed


finetuned: 100%|██████████| 25/25 [01:46<00:00,  4.27s/it]


>>> finetuned accuracy: 0.8550 (171/200)

                                                  precision    recall  f1-score   support

                           Refund_not_showing_up     1.0000    1.0000    1.0000         2
                                activate_my_card     1.0000    1.0000    1.0000         3
                                       age_limit     1.0000    1.0000    1.0000         3
                         apple_pay_or_google_pay     1.0000    1.0000    1.0000         3
                                     atm_support     1.0000    0.5000    0.6667         2
                                automatic_top_up     1.0000    0.6667    0.8000         3
         balance_not_updated_after_bank_transfer     1.0000    0.5000    0.6667         2
balance_not_updated_after_cheque_or_cash_deposit     1.0000    0.6667    0.8000         3
                         beneficiary_not_allowed     1.0000    1.0000    1.0000         3
                                 cancel_transfer     1.0


c:\Users\VINH\miniconda3\envs\manga_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\VINH\miniconda3\envs\manga_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\VINH\miniconda3\envs\manga_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is",

VRAM freed: 0.06GB used
=== SUMMARY ===
  zero_shot    0.6400  (64.00%)
  finetuned    0.8550  (85.50%)


## 6. Evaluate on full test set

Run zero_shot and finetuned on all 3,080 samples.

In [ ]:
print("\n" + "=" * 60)
print(f"FULL EVALUATION ({len(test_df)} samples)")
print("=" * 60)

full_results = {}
for mode in ("zero_shot", "finetuned"):
    clf = IntentClassification(f"{WORKING_DIR}/configs/inference.yaml", mode=mode)
print("=== FULL EVALUATION SUMMARY ===")
for mode, acc in full_results.items():
    print(f"  {mode:<12} {acc:.4f}  ({acc*100:.2f}%)")



FULL EVALUATION (3080 samples)
[ZERO_SHOT] Loading model: unsloth/Qwen3-4B-unsloth-bnb-4bit
==((====))==  Unsloth 2026.4.6: Fast Qwen3 patching. Transformers: 4.57.0.
   \\   /|    NVIDIA GeForce RTX 3070 Laptop GPU. Num GPUs = 1. Max memory: 8.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.1+cu118. CUDA: 8.6. CUDA Toolkit: 11.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/Qwen3-4B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
LangSmith tracing enabled — project: banking-intent-unsloth


zero_shot:  41%|████      | 157/385 [3:51:28<6:21:45, 100.46s/it] 